In [1]:
from instances.instances import generate_instances

instances = generate_instances(filename="TSP50.pkl", instance_count=1000, cities=50, seed=42)

In [2]:
from data.generation import generate_train_data
from data.adapters.input.sparse import SparseInputAdapter
from data.adapters.output.basic import BasicOutputAdapter

input_config = (SparseInputAdapter, 50)
output_config = (BasicOutputAdapter, 50)

generate_train_data(
    instance_file="TSP50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    num_workers=12,
    size=10000
)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 10000)


In [3]:
from data.preprocessing import load_dataset, split_dataset

train_file, val_file = split_dataset("train_data.h5", 8000)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (8000 muestras) en: train_data_train.h5
Guardando Test puro (2000 muestras) en: train_data_test.h5
Dataset train_data_train.h5 cargado con 8000 muestras.
Dataset train_data_test.h5 cargado con 2000 muestras.


In [5]:
from models.gat import TSP_GATModel

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
model = TSP_GATModel(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout
)

In [6]:
from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

model = sl_train(
    model=model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    weight_decay=1e-5,
    loss_fn=CrossEntropyLoss(),
    patience=10,
    metrics=[Accuracy()],
    metrics_filename="metrics_gat.txt",
    seed=42
)

** Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 1.9725 | Val CrossEntropy: 1.2826
    Accuracy: 62.60%
Epoch 2/10
    Train CrossEntropy: 1.0755 | Val CrossEntropy: 0.8909
    Accuracy: 74.20%
Epoch 3/10
    Train CrossEntropy: 0.8467 | Val CrossEntropy: 0.7538
    Accuracy: 78.60%
Epoch 4/10
    Train CrossEntropy: 0.7526 | Val CrossEntropy: 0.6824
    Accuracy: 79.95%
Epoch 5/10
    Train CrossEntropy: 0.6941 | Val CrossEntropy: 0.6369
    Accuracy: 82.00%
Epoch 6/10
    Train CrossEntropy: 0.6533 | Val CrossEntropy: 0.6050
    Accuracy: 81.30%
Epoch 7/10
    Train CrossEntropy: 0.6296 | Val CrossEntropy: 0.5858
    Accuracy: 82.20%
Epoch 8/10
    Train CrossEntropy: 0.6081 | Val CrossEntropy: 0.5679
    Accuracy: 82.45%
Epoch 9/10
    Train CrossEntropy: 0.5879 | Val CrossEntropy: 0.5664
    Accuracy: 82.60%
Epoch 10/10
    Train CrossEntropy: 0.5822 | Val CrossEntropy: 0.5499
    Accuracy: 82.30%

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-